# Biohub Full-Resolution Subset Downloader (v2) - ZSNS005

**Goal:** get **true Level-0 (full-resolution) DaXi microscopy pixels** from the CZ Biohub
ZebraHub volumes into Kaggle, *without* the Level-2 downsampling handicap of the old script,
while respecting Kaggle's storage limits.

> **v2 fix (chunk keys):** v1 wrote every chunk under a corrupted zip key
> (`0/330/0000` instead of `0/330/0/0/0/0` - missing `/` separators during directory
> recursion), so zarr opened the store fine but every chunk read came back as
> `fill_value=0` (all-black patches, `nonzero=0` in VERIFY). Enumeration now joins path
> components with explicit `/`. Re-download from a clean output dir (do not resume v1
> zips/manifests - the key scheme differs). VERIFY must print `nonzero > 0` on at least
> 2 of the 3 sampled patches.

> **v2.2 hardening (black-zip guard):** STEP 2 now detects a legacy v1 zip in the output
> dir (malformed chunk keys) and restarts it cleanly instead of resuming, and VERIFY
> hard-fails the run on malformed keys or on zip-black/server-nonzero chunk reads (live
> cross-check) - so an all-black zip can no longer be published or silently resumed.
> **v2.3 fix (double-slash keys + self-heal rekey):** the Biohub server lists directories
> with a trailing slash in its JSON listings (`0/`, `2/`, ...), so v2.2 runs wrote chunk
> keys like `0/110/0//2//0//1` - fully valid chunk bytes under a key zarr can never
> request (the v2.2 key audit catches this before publish). Listing names are now
> stripped/collapsed and walk() refuses any malformed key; if an existing zip carries the
> double-slash signature, STEP 2 REKEYS it in place (bytes kept, no re-download) and
> rewrites the manifest keys so resume still skips completed chunks.


### Why subsets? The measured reality (probed live, 2026-09)

| Volume | Full L0 on server | Files | Verdict |
|---|---|---|---|
| ZSNS001 | **~187 GB** | 155k | full volume cannot fit anywhere on Kaggle |
| ZSNS002 | **~370 GB** | 346k | idem (100 GB is the hard per-dataset cap) |
| ZSNS003 | ~49 GB | ~1k | fits only as its own 100GB-class dataset |
| ZSNS004 | ~81 GB | ~3.6k | idem |
| ZSNS005 | ~96 GB | ~3.6k | idem |

The old claim of "~15 GB per full-res volume" was wrong by ~12x. Total full L0 for the 5
volumes is ~780 GB. **No Kaggle artifact can hold that** (datasets cap at 100 GB, notebook
output at 19.5 GB).

### The strategy that DOES work

1. **Full resolution, subset of timepoints.** We download consecutive-frame *pairs*
   `(t, t+1)` every `PAIR_STRIDE` timepoints — exactly the `(t, t+1)` structure the
   edge-predictor trainer needs. Pixels are 100% Level-0; we simply do not take every frame.
2. **One Kaggle dataset per volume**, each auto-planned to fit under the 19.5 GB
   notebook-output cap (default budget 18 GB), then attached together in the trainer.
3. **Single `.zip` store per volume.** A zarr store zipped with `ZIP_STORED` is directly
   readable by `zarr.ZipStore` — one big file uploads/attaches fast and avoids 100k-file
   datasets. Missing (un-downloaded) chunks simply read back as zeros (`fill_value`),
   which was validated against the real server.
4. **Resume-safe.** A `manifest.jsonl` records every completed chunk; re-running the
   notebook skips them. The zip is checkpointed (closed/reopened) every ~300 MB so a crash
   never corrupts it.

### Runbook (5 short runs) - **this copy is pre-set to ZSNS005**

| Run | set `VOLUME` | produces dataset |
|---|---|---|
| 1 | `ZSNS001` | `biohub-zsns001-l0-subset` |
| 2 | `ZSNS002` | `biohub-zsns002-l0-subset` |
| 3 | `ZSNS003` | `biohub-zsns003-l0-subset` |
| 4 | `ZSNS004` | `biohub-zsns004-l0-subset` |
| 5 | `ZSNS005` | `biohub-zsns005-l0-subset` |

Each run: *Run All* → ~25-45 min (server throttles ~100-200 KB/s per connection; we use
~96 parallel connections for ~10-13 MB/s) → then publish (UI or API, last cell).
Finally attach all 5 datasets (+ the Freitas synthetic dataset) in the trainer notebook.

### Server facts this notebook relies on (validated live)

- `public.czbiohub.org` is a **Caddy file server**; directory listings are JSON when requested
  with header `Accept: application/json` (entries carry exact byte sizes and relative URLs).
- OME-ZARR v0.4 stores, `dimension_separator="/"`, blosc-lz4, uint16, axes `t,c,z,y,x`
  (ZSNS003 uses uppercase names — we index positionally, so it does not matter).
- `Range` requests return `206` — chunk-level resume works.
- The **root** of `/royerlab/ultrack/` is a *custom HTML page* (not an autoindex!) — the old
  script's regex fails there. We always hit concrete `.ome.zarr/` subpaths, which are proper
  JSON-listable directories.
- Voxel scales (read live from `.zattrs`): e.g. ZSNS001 L0 = 1.24 µm (z) × 0.439 µm (y,x).

In [ ]:
# ============================== CONFIG ==============================
# Edit VOLUME for each of the 5 runs: ZSNS001 ... ZSNS005
VOLUME         = "ZSNS005"          # pre-set for this notebook copy (one copy per volume)


PAIR_STRIDE    = 10               # int >= 2, or "auto" (fit to budget via consecutive-frame pairs)
T_RANGE        = None               # optional (t0, t1) to restrict time range, e.g. (0, 400)
BUDGET_GB      = 19.5               # hard target; keep <= 19.5 (Kaggle /kaggle/working cap)
LEVELS_PRIORITY = ["0"]             # L0 only; L1/L2 were always trimmed, skip enumeration
FETCH_TRACK_CSV = True              # also fetch the ZSNSxxx track CSV required by the trainer
WORKERS        = 96                 # parallel connections (96-128 measured optimal)
OUT_ROOT       = "/kaggle/working"  # output dir (dataset will be created from its contents)
PUBLISH        = "none"             # "none" | "api"  (api = push dataset via kaggle CLI in last cell)
DATASET_PUBLIC = True               # when publishing via API; public datasets bypass the 100GB private quota
                                 # NOTE: check competition rules before making training data public!

print(f"Volume: {VOLUME} | stride: {PAIR_STRIDE} | budget: {BUDGET_GB} GB | workers: {WORKERS}")

In [ ]:
# ============================== CONSTANTS + HELPERS ==============================
import json, os, re, sys, time, hashlib, threading, urllib.request, urllib.error
from pathlib import Path
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

ZH = "https://public.czbiohub.org/royerlab/zebrahub/imaging/single-objective"
VOLUMES = {
    "ZSNS001": f"{ZH}/ZSNS001.ome.zarr",
    "ZSNS002": f"{ZH}/ZSNS002.ome.zarr",
    "ZSNS003": f"{ZH}/ZSNS003.ome.zarr",
    "ZSNS004": f"{ZH}/ZSNS004.ome.zarr",
    "ZSNS005": f"{ZH}/ZSNS005.ome.zarr",
}
# measured live (Sept 2026): median L0 MB per timepoint + T; used only as a PRIOR for auto-stride
VOLUME_PRIOR = {
    "ZSNS001": dict(T=791,  per_t_mb=236.0),
    "ZSNS002": dict(T=1100, per_t_mb=337.0),
    "ZSNS003": dict(T=515,  per_t_mb=95.0),
    "ZSNS004": dict(T=600,  per_t_mb=136.0),
    "ZSNS005": dict(T=600,  per_t_mb=160.0),
}

BASE = VOLUMES[VOLUME]
OUT  = Path(OUT_ROOT); OUT.mkdir(parents=True, exist_ok=True)
ZIP_PATH   = OUT / f"{VOLUME}.ome.zarr.zip"
MANIFEST   = OUT / "manifest.jsonl"
ENUM_CACHE = OUT / "enumerate.json"
PLAN_JSON  = OUT / "plan.json"

UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"}

def http_get(url, timeout=90, retries=4, rng=None):
    """GET with retries; returns (status, bytes) - status 200/206/404."""
    for attempt in range(1, retries + 1):
        try:
            h = dict(UA)
            if rng: h["Range"] = f"bytes={rng[0]}-{rng[1]}"
            req = urllib.request.Request(url, headers=h)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, r.read()
        except urllib.error.HTTPError as e:
            if e.code in (404, 410):
                return e.code, b""
            time.sleep(1.2 * attempt)
        except Exception:
            time.sleep(1.5 * attempt)
    return -1, b""

def jlist(url, retries=4):
    """Caddy JSON directory listing -> [{name, url, size, is_dir, ...}] (absolute urls)."""
    if not url.endswith("/"): url += "/"
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={**UA, "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=45) as r:
                entries = json.loads(r.read().decode())
            # FIX (v2.3): this server lists DIRECTORIES with a trailing slash in 'name'
            # (verified live 2026-09-18: entries come back as '0/', '1/', ...). Fed into
            # walk() verbatim, that produced zip keys like '0/110/0//2//0//1' - valid
            # chunk bytes under a key zarr can never request. Strip the slashes here so
            # every key built downstream is clean regardless of the server's format.
            return [(e["name"].rstrip("/"), urljoin(url, e["url"]), e.get("is_dir", False),
                     e.get("size"))
                    for e in entries if e["name"] not in ("../", "..", "./", ".")]
        except Exception:
            if attempt == retries:
                raise RuntimeError(f"listing failed: {url}")
            time.sleep(1.0 * attempt)

def human(n):
    n = float(n)
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or u == "TB": return f"{n:.2f} {u}"
        n /= 1024

print("helpers ready |", VOLUME, "->", BASE)

In [ ]:
# ============================== STEP 1: PLAN ==============================
# Reads .zattrs/.zarray, probes a few timepoints, picks the stride, then enumerates
# the EXACT chunk list for the selected timepoints (sizes included -> exact budget).

def get_meta():
    st, b = http_get(f"{BASE}/.zattrs");  assert st == 200, ".zattrs missing"
    zattrs = json.loads(b)
    ms = zattrs["multiscales"][0]
    levels = [d["path"] for d in ms["datasets"]]
    scales = {d["path"]: d.get("coordinateTransformations", [{}])[0].get("scale")
              for d in ms["datasets"]}
    zarrays = {}
    for lv in levels:
        st, b = http_get(f"{BASE}/{lv}/.zarray"); assert st == 200, f"{lv}/.zarray missing"
        zarrays[lv] = json.loads(b)
    return zattrs, levels, scales, zarrays

zattrs, AVAIL_LEVELS, SCALES, ZARRAYS = get_meta()
T = ZARRAYS["0"]["shape"][0]
print(f"axes: {[a.get('name') for a in zattrs['multiscales'][0]['axes']]}")
print(f"levels on server: {AVAIL_LEVELS}")
for lv in AVAIL_LEVELS:
    za = ZARRAYS[lv]
    print(f"  L{lv}: shape={za['shape']} chunks={za['chunks']} dtype={za['dtype']} "
          f"sep={za.get('dimension_separator')!r} scale={SCALES[lv]}")

# --- probe a few timepoints to estimate per-t bytes at L0 ---
def t_bytes(t):
    total, nf = 0, 0
    def walk(u):
        nonlocal total, nf
        for name, eu, is_dir, size in jlist(u):
            if is_dir: walk(eu)
            else: total += size or 0; nf += 1
    walk(f"{BASE}/0/{t}/")
    return nf, total

probe_ts = sorted(set([0, T // 2, T - 1] + ([T // 4] if T > 4 else [])))
with ThreadPoolExecutor(min(8, len(probe_ts))) as ex:
    probed = list(ex.map(t_bytes, probe_ts))
per_t_prior = VOLUME_PRIOR[VOLUME]["per_t_mb"] * 1e6
per_t_probe = sum(p[1] for p in probed) / len(probed)
per_t = max(per_t_probe, per_t_prior * 0.5)  # trust live probe, floor at half the prior
print(f"probe t={probe_ts}: files={[p[0] for p in probed]} MB={[round(p[1]/1e6,1) for p in probed]}")
print(f"per-t estimate: {per_t/1e6:.1f} MB (probe) vs prior {per_t_prior/1e6:.1f} MB")

# --- choose pair stride ---
keep_levels = [lv for lv in LEVELS_PRIORITY if lv in AVAIL_LEVELS] or ["0"]
# pyramid factor: each level is ~8x smaller (2x per axis x3); use measured chunk counts when possible
lvl_factor = {}
for lv in keep_levels:
    za = ZARRAYS[lv]
    import math
    grid = math.prod(math.ceil(s / c) for s, c in zip(za["shape"], za["chunks"]))
    grid0 = math.prod(math.ceil(s / c) for s, c in zip(ZARRAYS["0"]["shape"], ZARRAYS["0"]["chunks"]))
    lvl_factor[lv] = max(grid / grid0, 1e-4)   # file-count ratio ~ byte ratio for same dtype
factor_sum = sum(lvl_factor.values())

t_lo, t_hi = (0, T - 1) if not T_RANGE else (max(0, T_RANGE[0]), min(T - 1, T_RANGE[1]))
budget_bytes = BUDGET_GB * 1e9
if PAIR_STRIDE == "auto":
    n_t_max = int(budget_bytes / (per_t * factor_sum))
    n_pairs = max(1, n_t_max // 2)
    stride = max(2, math.ceil((t_hi - t_lo + 1) / n_pairs))
else:
    stride = int(PAIR_STRIDE)
starts = list(range(t_lo, t_hi - 1, stride))            # pair anchors (need t and t+1)
selected_t = sorted(set([t for s in starts for t in (s, s + 1) if t <= t_hi]))
print(f"stride={stride} -> {len(starts)} consecutive-frame pairs, {len(selected_t)} timepoints: "
      f"t={selected_t[:6]}...{selected_t[-3:]}")

# --- EXACT enumeration of selected timepoints (cached for resume) ---
def enumerate_level(lv):
    """Returns dict key -> (url, size) for the selected t's of one level + level metadata.

    FIX (v2 - the silent-zeros bug): v1 built keys as prefix + name with the top-level
    prefix ending in '/' and never re-inserted the separator during recursion, so chunk
    keys were written as '0/330/0000' instead of '0/330/0/0/0/0'. The zip then opened
    fine (.zarray keyed correctly) but every chunk lookup missed -> zarr silently
    returned fill_value=0 and every patch read as all-black. Keys are now joined with
    explicit '/' separators. NOTE: do NOT resume v1 zips/manifests (key scheme differs);
    start from a clean output dir.
    """
    tasks = {}
    def walk(u, prefix):
        for name, eu, is_dir, size in jlist(u):
            name = name.rstrip("/")                               # FIX (v2.3): '0/' -> '0'
            child = name if not prefix else prefix + "/" + name   # FIX: always join with '/'
            child = re.sub(r"/+", "/", child)                     # FIX (v2.3): collapse doubles
            if is_dir:
                walk(eu, child)
            else:
                # FIX (v2.3): never write a malformed key - it would silently read as
                # fill_value 0 (the exact black-zip failure mode this notebook guards).
                if not re.fullmatch(re.escape(str(lv)) + r"/\d+(?:/\d+)*", child):
                    raise RuntimeError(
                        f"malformed chunk key {child!r} for L{lv} - refusing to add it to "
                        f"the download plan (a malformed key silently reads as fill 0).")
                tasks[child] = (eu, size)
    def one_t(t):
        walk(f"{BASE}/{lv}/{t}/", f"{lv}/{t}")                    # FIX: no trailing '/' in prefix
        return t
    with ThreadPoolExecutor(16) as ex:
        list(ex.map(one_t, selected_t))
    st, b = http_get(f"{BASE}/{lv}/.zarray")
    if st == 200: tasks[f"{lv}/.zarray"] = (f"{BASE}/{lv}/.zarray", len(b))
    return tasks

enum_all = {}
t0 = time.time()
for lv in keep_levels:
    enum_all[lv] = enumerate_level(lv)
    print(f"enumerated L{lv}: {len(enum_all[lv])} files, "
          f"{human(sum(v[1] or 0 for v in enum_all[lv].values()))} ({time.time()-t0:.0f}s)")
for mf in (".zgroup", ".zattrs"):
    st, b = http_get(f"{BASE}/{mf}")
    if st == 200: enum_all.setdefault("meta", {})[mf] = (f"{BASE}/{mf}", len(b))

json.dump({lv: {k: [u, s] for k, (u, s) in d.items()} for lv, d in enum_all.items()},
          open(ENUM_CACHE, "w"))
plan = {
    "volume": VOLUME, "base": BASE, "stride": stride, "pair_anchors": starts,
    "selected_t": selected_t, "levels": keep_levels, "scales": SCALES,
    "level_shapes": {lv: ZARRAYS[lv]["shape"] for lv in keep_levels},
    "files": {lv: len(d) for lv, d in enum_all.items()},
    "bytes": {lv: sum(v[1] or 0 for v in d.values()) for lv, d in enum_all.items()},
}
total = sum(plan["bytes"].values())
print(f"\nPLAN: {sum(plan['files'].values())} files, {human(total)} total")
for lv in keep_levels + ["meta"]:
    if lv in plan["files"]:
        print(f"  L{lv}: {plan['files'][lv]:6d} files  {human(plan['bytes'][lv])}")

# --- budget guard: drop deepest non-L0 level, then trailing pairs, if needed ---
def trim_plan():
    global enum_all, starts, selected_t, total
    lvl_order = [lv for lv in keep_levels if lv != "0"]     # extras, deepest first
    for lv in sorted(lvl_order, key=float, reverse=True):
        if total <= budget_bytes: break
        enum_all.pop(lv, None); print(f"  ! dropped level {lv} to fit budget")
    while total > budget_bytes and len(starts) > 1:
        s = starts.pop(); selected_t = sorted(set([t for x in starts for t in (x, x + 1) if t <= t_hi]))
        for lv in list(enum_all.keys()):
            if lv == "meta": continue
            d = enum_all[lv]
            for k in [k for k in d if k.split("/")[1] in (str(s), str(s + 1))]:
                d.pop(k, None)
        total = sum(sum(v[1] or 0 for v in d.values()) for d in enum_all.values())
    return total

if total > budget_bytes * 1.02:
    print(f"\n! over budget ({human(total)} > {BUDGET_GB} GB) - trimming...")
    total = trim_plan()
    print(f"after trim: {human(total)}")
assert total <= budget_bytes * 1.02, f"plan still over budget: {human(total)}"
plan.update({"files": {lv: len(d) for lv, d in enum_all.items()},
             "bytes": {lv: sum(v[1] or 0 for v in d.values()) for lv, d in enum_all.items()},
             "total_bytes": total})
json.dump(plan, open(PLAN_JSON, "w"), indent=1)
print(f"\nplan saved -> {PLAN_JSON}")

In [ ]:
# ============================== STEP 2: DOWNLOAD (streaming zip writer) ==============================
# Chunks are fetched with WORKERS parallel connections and streamed into ONE zip
# (ZIP_STORED, zarr-readable). manifest.jsonl makes re-runs resume cleanly.
import zipfile

done = set()
if MANIFEST.exists():
    for line in open(MANIFEST):
        try:
            r = json.loads(line)
            if r.get("status") == "ok": done.add(r["key"])
        except Exception: pass
print(f"resume: {len(done)} chunks already recorded in manifest")

# FIX (v2.2/v2.3): legacy-zip guard. Two key-corruption signatures exist:
#   (a) v1: '0/330/0000' - indices concatenated without separators; not reliably
#       invertible -> delete zip+manifest and start clean.
#   (b) v2.2-on-this-server: Caddy lists DIRECTORIES with a trailing slash ('0/'), so
#       keys were written as '0/110/0//2//0//1'. The chunk BYTES are fully valid (the
#       server even serves '//' URLs, verified live) - only the key string is wrong:
#       REKEY the zip + manifest in place instead of throwing away ~18 GB of downloads.
if ZIP_PATH.exists():
    import shutil as _shutil
    try:
        with zipfile.ZipFile(ZIP_PATH) as _z0:
            _names0 = _z0.namelist()
        _ck0 = [n for n in _names0 if re.match(r"^0/\d+/", n)]
        _mal0 = [n for n in _ck0 if len(n.split("/")) != 6]
    except Exception:
        _ck0, _mal0 = [], []
    if _ck0 and _mal0:
        def _rekey(n):
            return re.sub(r"/+", "/", n)
        _fixable = [n for n in _mal0 if "//" in n and len(_rekey(n).split("/")) == 6]
        _need = int(ZIP_PATH.stat().st_size * 1.02) + (1 << 30)
        if len(_fixable) == len(_mal0) and _shutil.disk_usage(str(OUT)).free > _need:
            print(f"! existing zip holds {len(_mal0)}/{len(_ck0)} double-slash keys "
                  f"(server lists dirs as '0/') -> REKEYING in place (bytes are valid)")
            try:
                _tmp = str(ZIP_PATH) + ".rekeying"
                _seen = set()
                with zipfile.ZipFile(ZIP_PATH) as _zin, \
                     zipfile.ZipFile(_tmp, "w", zipfile.ZIP_STORED, allowZip64=True) as _zout:
                    for _info in _zin.infolist():
                        _nk = _rekey(_info.filename)
                        assert _nk not in _seen, f"rekey collision: {_info.filename} -> {_nk}"
                        _seen.add(_nk)
                        _zout.writestr(_nk, _zin.read(_info.filename))
                with zipfile.ZipFile(_tmp) as _zt:
                    _bad = _zt.testzip()
                if _bad is not None:
                    raise RuntimeError(f"CRC failure at {_bad}")
                os.replace(_tmp, str(ZIP_PATH))
                if MANIFEST.exists():
                    _lines = []
                    for _line in open(MANIFEST):
                        try:
                            _r = json.loads(_line); _r["key"] = _rekey(_r["key"])
                            _lines.append(json.dumps(_r))
                        except Exception:
                            _lines.append(_line.rstrip("\n"))
                    MANIFEST.write_text("\n".join(_lines) + "\n")
                done = set()
                for _line in open(MANIFEST):
                    try:
                        _r = json.loads(_line)
                        if _r.get("status") == "ok": done.add(_r["key"])
                    except Exception: pass
                print(f"rekeyed {len(_seen)} entries, manifest keys rewritten, "
                      f"resume set = {len(done)} chunks -> continuing without re-download")
            except Exception as _e:
                print(f"! rekey failed ({type(_e).__name__}: {_e}) -> deleting zip AND "
                      f"manifest, starting clean")
                _tmp2 = Path(str(ZIP_PATH) + ".rekeying")
                if _tmp2.exists(): _tmp2.unlink()
                ZIP_PATH.unlink(); MANIFEST.unlink()
                if ENUM_CACHE.exists(): ENUM_CACHE.unlink()
                done = set()
        else:
            print(f"! existing zip holds {len(_mal0)}/{len(_ck0)} v1-style malformed chunk keys "
                  f"(e.g. {_mal0[:2]}) -> deleting zip AND manifest, starting clean")
            ZIP_PATH.unlink(); MANIFEST.unlink()
            if ENUM_CACHE.exists(): ENUM_CACHE.unlink()
            done = set()


all_tasks = []           # (key, url, expected_size)
for lv, d in enum_all.items():
    for k, (u, s) in d.items():
        all_tasks.append((k, u, s))
all_tasks.sort()
todo = [t for t in all_tasks if t[0] not in done]
print(f"to download: {len(todo)} / {len(all_tasks)} files, {human(sum(t[2] or 0 for t in todo))}")

stat = {"files": 0, "bytes": 0, "fail": 0, "absent": 0, "zip_bytes": 0}
lock = threading.Lock()
mf = open(MANIFEST, "a")

def fetch_chunk(item):
    key, url, expected = item
    st, data = http_get(url, retries=5)
    if st == 404:                       # chunk never stored (sparse zarr) -> fill_value on read
        return key, None, "absent"
    if st != 200:
        return key, None, "failed"
    if expected and len(data) != expected:
        return key, None, "size-mismatch"
    return key, data, "ok"

class Ticker(threading.Thread):
    def __init__(self, total_n, total_b):
        super().__init__(daemon=True)
        self.total_n, self.total_b, self.t0, self.stop = total_n, total_b, time.time(), False
    def run(self):
        while not self.stop:
            with lock:
                n, b, f, a = stat["files"], stat["bytes"], stat["fail"], stat["absent"]
            el = time.time() - self.t0
            rate = b / el if el > 1 else 0
            eta = (self.total_b - b) / rate if rate > 0 else 0
            print(f"  [{VOLUME}] {n}/{self.total_n} files | {human(b)} | {rate/1e6:.1f} MB/s "
                  f"| ETA {int(eta//60):02d}:{int(eta%60):02d} | fail={f} absent={a}", flush=True)
            time.sleep(20)

CHECK_EVERY = 300 * 1024 * 1024      # reopen zip every ~300MB -> crash-safe checkpoint
since_check = 0
t0 = time.time()
tk = Ticker(len(todo), sum(t[2] or 0 for t in todo)); tk.start()

mode = "a" if ZIP_PATH.exists() and len(done) else "w"
if mode == "a":
    try:
        import zipfile as _zf; _zf.ZipFile(ZIP_PATH, "r").namelist()   # validate
    except Exception:
        print("! existing zip unreadable -> restarting zip AND manifest from scratch")
        ZIP_PATH.unlink(); mf.close(); MANIFEST.unlink(); mf = open(MANIFEST, "w")
        done, todo, mode = set(), [t for t in all_tasks], "w"

zf = zipfile.ZipFile(ZIP_PATH, mode, zipfile.ZIP_STORED, allowZip64=True)
try:
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(fetch_chunk, it): it for it in todo}
        for fut in as_completed(futs):
            key, data, status = fut.result()
            with lock:
                if status == "ok":
                    zf.writestr(key, data)
                    stat["files"] += 1; stat["bytes"] += len(data); stat["zip_bytes"] += len(data)
                    mf.write(json.dumps({"key": key, "bytes": len(data), "status": "ok"}) + "\n")
                    since_check += len(data)
                    if since_check > CHECK_EVERY:
                        zf.close(); zf = zipfile.ZipFile(ZIP_PATH, "a", zipfile.ZIP_STORED, allowZip64=True)
                        since_check = 0; mf.flush()
                elif status == "absent":
                    stat["absent"] += 1
                    mf.write(json.dumps({"key": key, "bytes": 0, "status": "absent"}) + "\n")
                else:
                    stat["fail"] += 1
                    mf.write(json.dumps({"key": key, "bytes": 0, "status": status}) + "\n")
finally:
    zf.close(); mf.flush(); mf.close(); tk.stop = True

el = time.time() - t0
print(f"\nDONE in {int(el//60)}m{int(el%60):02d}s | {stat['files']} chunks, {human(stat['bytes'])} "
      f"({stat['bytes']/max(el,1)/1e6:.1f} MB/s) | absent={stat['absent']} fail={stat['fail']}")
print(f"zip: {human(ZIP_PATH.stat().st_size)} -> {ZIP_PATH}")
json.dump({"volume": VOLUME, "stats": stat, "zip": str(ZIP_PATH),
           "zip_size": ZIP_PATH.stat().st_size, "elapsed_s": int(el)},
          open(OUT / "DONE.json", "w"), indent=1)
assert stat["fail"] < max(10, 0.001 * len(todo)), f"too many failed chunks ({stat['fail']}) - re-run this cell to resume"

In [ ]:
# ============================== STEP 2b: TRACK CSV ==============================
# Fetch the real ZSNS track labels required by the appearance trainer.
# FIX (v2.1 - ZSNS002 404): the server publishes *_tracks.csv for ZSNS001/003/004/005
# only - there is NO ZSNS002 track CSV anywhere under public.czbiohub.org (verified
# live: the single-objective/ listing has no ZSNS002 CSV, only its neuroglancer .json).
# v1 raised HTTPError here and killed the run AFTER the zip was complete but BEFORE
# the STEP 3 verify cell. This version tries several candidate URLs and degrades
# gracefully: a missing CSV is a loud warning + a TRACK_CSV_MISSING.txt note in the
# output, never a crash. The trainer simply gets no label pairs for that volume.
TRACK_CSV_PATH = OUT / f"{VOLUME}_tracks.csv"

_CANDIDATE_CSVS = [
    f"{ZH}/{VOLUME}_tracks.csv",           # standard location (001/003/004/005)
    f"{ZH}/{VOLUME}_tracks_filtered.csv",  # possible alternate name
    f"{ZH}/{VOLUME.lower()}_tracks.csv",   # case variant
]

if TRACK_CSV_PATH.exists():
    print(f"track CSV already present -> {TRACK_CSV_PATH} ({human(TRACK_CSV_PATH.stat().st_size)})")
elif FETCH_TRACK_CSV:
    import urllib.request, shutil
    got = False
    for _url in _CANDIDATE_CSVS:
        try:
            print(f"trying {_url} ...")
            req = urllib.request.Request(_url, headers=UA)
            with urllib.request.urlopen(req, timeout=900) as r, open(TRACK_CSV_PATH, "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 22)  # 4 MB chunks
            if TRACK_CSV_PATH.stat().st_size < 1_000_000:   # real CSVs are 200-900 MB
                print(f"  ! suspiciously small file ({human(TRACK_CSV_PATH.stat().st_size)}) -> treating as missing")
                TRACK_CSV_PATH.unlink(missing_ok=True)
                continue
            print(f"track CSV: {human(TRACK_CSV_PATH.stat().st_size)} -> {TRACK_CSV_PATH}")
            got = True
            break
        except urllib.error.HTTPError as e:
            print(f"  HTTP {e.code} -> trying next candidate")
            TRACK_CSV_PATH.unlink(missing_ok=True)
        except Exception as e:
            print(f"  {type(e).__name__}: {e} -> trying next candidate")
            TRACK_CSV_PATH.unlink(missing_ok=True)
    if not got:
        _warn = (f"NO track CSV published for {VOLUME} on the Biohub server "
                 f"(verified live: single-objective/ lists *_tracks.csv for ZSNS001/003/004/005 "
                 f"only). The {VOLUME}.ome.zarr.zip is still complete and valid. Without labels "
                 f"the trainer cannot build time-pairs from this volume - attaching this "
                 f"dataset to the trainer is optional.")
        print(f"\n! WARNING: {_warn}\n")
        (OUT / "TRACK_CSV_MISSING.txt").write_text(_warn + "\n")
else:
    print("FETCH_TRACK_CSV=False -> skipping real track CSV download")


In [ ]:
# ============================== STEP 3: VERIFY (zarr read-back) ==============================
# Installs pinned zarr and opens the freshly-built zip exactly like the trainer will.

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "zarr==2.18.4", "numcodecs==0.15.1"], check=False)
import zarr, numpy as np

def _level_ok(root, k):
    try:
        root[k]; return True
    except Exception:
        return False

zs = zarr.ZipStore(str(ZIP_PATH), mode="r")
root = zarr.open_group(store=zs, mode="r")
present = [k for k in AVAIL_LEVELS if _level_ok(root, k)]
print("store opens OK | levels present:", present)
a0 = root["0"]
print(f"L0: shape={a0.shape} chunks={a0.chunks} dtype={a0.dtype}")

# FIX (v2.2): hard gates - an all-black zip must NEVER reach the publish step.
# (a) KEY AUDIT: v1 wrote chunk keys as '0/330/0000' (3 components) instead of
#     '0/330/0/0/0/0' (6 components). Such zips open fine, list the right timepoints,
#     and read as fill_value 0 EVERYWHERE. One malformed key poisons reads -> abort.
import zipfile as _zf, re as _re, json as _json
from numcodecs import Blosc as _Blosc, Zstd as _Zstd, LZ4 as _LZ4
_DEC = {"blosc": _Blosc, "zstd": _Zstd, "lz4": _LZ4}
with _zf.ZipFile(str(ZIP_PATH)) as _z:
    _names = _z.namelist()
    _za0 = _json.loads(_z.read("0/.zarray"))
_ck = [n for n in _names if _re.match(r"^0/\d+/", n)]
_mal = [n for n in _ck if len(n.split("/")) != 6]
print(f"key audit: {len(_ck)} L0 chunk entries | malformed keys: {len(_mal)}")
if _ck and _mal:
    zs.close()
    _dbl = sum(1 for n in _mal if "//" in n)
    if _dbl == len(_mal):
        raise RuntimeError(
            f"VERIFY ABORT: {len(_mal)} of {len(_ck)} chunk keys carry the v2.2 "
            f"'double-slash' signature, e.g. {_mal[:2]} (the server lists directories as "
            f"'0/'). The chunk BYTES in this zip are valid and recoverable: just re-run "
            f"this notebook from the top - STEP 2 detects the zip and REKEYS it in place "
            f"(no re-download), then VERIFY passes and you can publish.")
    raise RuntimeError(
        f"VERIFY ABORT: {len(_mal)} of {len(_ck)} chunk keys are malformed, e.g. {_mal[:2]}. "
        f"That is the v1 key-corruption signature - this zip reads all-black. Delete the zip "
        f"AND manifest.jsonl and re-download from a CLEAN output dir.")

# (b) READ-BACK probes: deterministic t sample, ALWAYS including t=1 (the frame the
#     trainer's smoke test reads first), volume center + one random spot per t.
sel = [t for t in selected_t if t < a0.shape[0]]
rng = np.random.default_rng(0)
probe_ts = sorted({int(t) for t in rng.choice(sel, size=min(3, len(sel)), replace=False)} |
                  {min(1, a0.shape[0] - 1)})
ok, n_patches = 0, 0
for t in probe_ts:
    zc, yc, xc = [s // 2 for s in a0.shape[2:]]
    spots = [(zc, yc, xc),
             (int(rng.integers(0, a0.shape[2])), int(rng.integers(0, a0.shape[3])),
              int(rng.integers(0, a0.shape[4])))]
    for z, y, x in spots:
        patch = a0[t, 0, max(0, z-16):z+16, max(0, y-32):y+32, max(0, x-32):x+32]
        n_patches += 1
        print(f"  t={t} @(z{z},y{y},x{x}): patch {patch.shape} min={patch.min()} "
              f"max={patch.max()} mean={patch.mean():.2f} nonzero={(patch>0).sum()}")
        ok += int(patch.max() > 0)
absent_t = 1 if 1 not in sel else 2
while absent_t in sel and absent_t < a0.shape[0]: absent_t += 10
if absent_t < a0.shape[0]:
    pz = a0[absent_t, 0, :8, :8, :8]
    print(f"  t={absent_t} (not downloaded): max={pz.max()} (expect 0 = fill_value)")

# (c) SERVER CROSS-CHECK: decode the same chunk key from the zip and from the live
#     server. Zip black where the server serves pixels is the definitive black-zip
#     signature (v1 keys or lost chunks) -> abort. Server 404/both-empty = pass.
_ch = _za0["chunks"]
_shp = _za0["shape"]
def _covering_key(t, z, y, x):
    for n in _ck:
        p = n.split("/")
        if len(p) != 6 or int(p[1]) != int(t):
            continue
        cz, cy, cx = int(p[3]), int(p[4]), int(p[5])
        if (cz * _ch[2] <= z < min((cz + 1) * _ch[2], _shp[2]) and
                cy * _ch[3] <= y < min((cy + 1) * _ch[3], _shp[3]) and
                cx * _ch[4] <= x < min((cx + 1) * _ch[4], _shp[4])):
            return n
    return None
def _nz_of(raw):
    comp = (_za0.get("compressor") or {}).get("id")
    if comp and comp in _DEC:
        raw = _DEC[comp]().decode(raw)
    return int((np.frombuffer(raw, dtype=_za0["dtype"]) > 0).sum())
_mismatch, _agreed_empty, _checked = 0, 0, 0
for t in probe_ts:
    zc, yc, xc = [s // 2 for s in a0.shape[2:]]
    _k = _covering_key(t, zc, yc, xc)
    if _k is None:
        continue
    _checked += 1
    with _zf.ZipFile(str(ZIP_PATH)) as _z:
        zip_nz = _nz_of(_z.read(_k))
    st, srv = http_get(f"{BASE}/{_k}")
    srv_nz = _nz_of(srv) if (st == 200 and srv) else 0
    if zip_nz > 0:
        tag = (f"zip={zip_nz} nonzero, server={srv_nz} nonzero - agree" if st == 200 else
               f"zip={zip_nz} nonzero (server HTTP {st} - unreachable/missing; zip has pixels)")
    elif srv_nz > 0:
        _mismatch += 1
        tag = f"ZIP BLACK ({zip_nz} nonzero) BUT SERVER SERVES {srv_nz} nonzero !!"
    else:
        _agreed_empty += 1
        tag = "both empty (plausible background)" if st == 200 else f"server HTTP {st}"
    print(f"  cross-check t={t} chunk {_k}: {tag}")
zs.close()
if _mismatch:
    raise RuntimeError(
        f"VERIFY ABORT: {_mismatch} checked chunk(s) read ALL-BLACK from the zip while the "
        f"server serves real pixels for the same keys. Do NOT publish - delete the zip and "
        f"manifest.jsonl and re-download from a CLEAN output dir.")
if ok == 0 and _agreed_empty == _checked and _checked > 0:
    print(f"\n! VERIFY WARNING: all probes read black, but the server agrees those chunks "
          f"are empty (sparse/background region). Inspect the patches above manually before "
          f"publishing.")
elif ok == 0:
    raise RuntimeError(
        f"VERIFY ABORT: all {n_patches} sampled patches (t={probe_ts}) read as ALL-ZERO. "
        f"Re-download from a CLEAN output dir - do not publish this zip.")
elif ok < (n_patches + 1) // 2:
    print(f"\n! VERIFY WARNING: only {ok}/{n_patches} patches non-empty - the embryo may "
          f"sit off-center; inspect the per-patch lines above before publishing.")
else:
    print(f"\nVERIFY PASSED ({ok}/{n_patches} patches non-empty across t={probe_ts})")

### STEP 4: Publish the dataset

**Path A — UI (recommended, no credentials needed):**
1. Top right → **Save Version** → **Save & Run All (Commit)**. Wait for the run to finish.
2. Open the committed version → **Output** tab → click the **⋮ menu / "New dataset"** button.
3. Title: `biohub-{volume}-l0-subset` (e.g. `biohub-zsns001-l0-subset`), keep it private or public.
4. Next run: change `VOLUME` at the top of this notebook and repeat (5 datasets total).

**Path B — API (does the same from inside the notebook):** set `PUBLISH = "api"` in CONFIG.
Requirements: *Add-ons → Secrets* → add a secret named `kaggle-json` whose value is the raw
content of your `kaggle.json` (kaggle.com → Account → Create New API Token).

In [ ]:
# ============================== STEP 4 (optional): PUBLISH VIA KAGGLE API ==============================
# Only runs when PUBLISH == "api". Expects secret "kaggle-json" (raw kaggle.json content),
# or an attached input dataset containing kaggle.json.

if PUBLISH == "api":
    import subprocess, pathlib
    token = None
    try:
        from kaggle_secrets import UserSecretClient
        token = UserSecretClient().get_secret("kaggle-json")
    except Exception as e:
        print("secrets unavailable:", e)
    if not token:
        for cand in pathlib.Path("/kaggle/input").rglob("kaggle.json"):
            token = cand.read_text(); print("using", cand); break
    assert token and '"username"' in token, "kaggle.json not found via secrets or inputs"

    kdir = pathlib.Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    (kdir / "kaggle.json").write_text(token)
    import os as _os; _os.chmod(kdir / "kaggle.json", 0o600)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)

    import json as _json
    meta = {"title": f"biohub-{VOLUME.lower()}-l0-subset",
            "id": _json.loads(token)["username"] + f"/biohub-{VOLUME.lower()}-l0-subset",
            "licenses": [{"name": "CC0-1.0"}],
            "private": not DATASET_PUBLIC}
    (OUT / "dataset-metadata.json").write_text(_json.dumps(meta, indent=1))
    r = subprocess.run(["kaggle", "datasets", "create", "-p", str(OUT)],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if "409" in (r.stderr or "") or "already exists" in (r.stderr or ""):
        print("dataset exists -> pushing as a new VERSION (replaces contents!)")
        r2 = subprocess.run(["kaggle", "datasets", "version", "-p", str(OUT), "-m", "update"],
                            capture_output=True, text=True)
        print(r2.stdout, r2.stderr)
    print(f"\nAttach this dataset in the trainer: search '{meta['title']}' under Add Input.")
else:
    print("PUBLISH != 'api' -> use Path A (UI) from the markdown cell above.")

### Notes & troubleshooting

- **v2.3 double-slash fix**: the server's JSON listings end directory names with '/'
  (`0/`), which made v2.2 write zip keys like `0/110/0//2//0//1` (the key audit aborts on
  these). The chunk bytes were always valid, so STEP 2 now self-heals: it rekeys the
  existing zip in place and rewrites the manifest keys - re-running the notebook costs
  minutes, not a fresh multi-GB download.
- **v2.2 black-zip guard**: VERIFY now hard-fails on malformed chunk keys or on
  zip-black/server-nonzero chunk reads (live cross-check), and STEP 2 auto-restarts when
  the output dir holds a legacy v1 zip (malformed keys). An all-black zip can no longer
  be published by accident - the 2026-09-17 incident (v1 zips trained on black pixels)
  is what this guards against.
- **ZSNS002 has no published track CSV** on the Biohub server (verified live:
  `single-objective/` lists `*_tracks.csv` for ZSNS001/003/004/005 only).
  STEP 2b now warns and continues instead of crashing - the zip is still
  complete, but without labels the trainer gets no pairs from ZSNS002, so
  attaching its dataset is optional.
- **"Failed" chunks**: just re-run the download cell — the manifest skips completed chunks,
  only failures/absent get retried.
- **Session died mid-run**: re-run the whole notebook with the same `VOLUME`. Enumeration is
  cached (`enumerate.json`), the manifest resumes the download. (If the zip itself got
  corrupted, the notebook detects it and restarts cleanly.)
- **Want every single timepoint** (no stride)? Set `PAIR_STRIDE = 2` *and* `BUDGET_GB = None`…
  but expect ~187-370 GB for ZSNS001/002 — that only makes sense outside Kaggle (e.g. a
  workstation or university cluster). On Kaggle, keep the budget.
- **Fidelity check**: the trainer notebook reads these zips with `zarr.ZipStore` and samples
  Level-0 patches around your nodes — pixel-exact, zero downsampling.
- The Freitas synthetic dataset (18.5 GB) is **not** downloaded here — attach
  `josefreitasalvesneto/biohub-synthetic-dataset` directly as a notebook input.